### Import Modules

In [5]:
import pandas as pd,numpy as np,json
from sklearn.metrics.pairwise import cosine_distances
import getpass,os
from langchain.chat_models import init_chat_model
from ai_patterns_mining import parse_json_safe,Config
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from time import time

### Setup Gemini

In [2]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

def generate_embedding(text: list[str])->list:
    response = embeddings.embed_documents(text,output_dimensionality=768)
    return response

vector_store = InMemoryStore(index={"embed": generate_embedding, "dims": 3072})
user_id = "wso2"
application_context = "tag-generator"
namespace = (user_id,application_context)

agent = create_agent(
    llm,
    checkpointer=InMemorySaver(),
)

In [3]:
prompt = """\
Please consider the following AI pattern and describe it using comma separated keywords. 
* If there already generated keywords in history for current pattern or relatively same pattern, keep exactly the same as previous keywords. 
* if there are similar pattern can be grouped together with current pattern, reuse the keywords from the most relevant similar pattern and add a new keyword to reflect the current pattern.
* Make sure the keywords are relevant to the pattern. Do not invent new keywords. Do not add keywords that are not direct part of the pattern.
* If there are no previous keywords, generate new relevant keywords based on the pattern content.

give output in the following format in JSON format:
keywords: keyword1, keyword2, keyword3, ...
thinking: there are n previous patterns with similar content, so I will reuse the keywords from the most relevant one.

Current Pattern:
{pattern}
"""

In [4]:
def generate_description(pattern: dict)->str:
    user_mg = prompt.format(pattern=json.dumps(pattern, indent=2))
    
    output = agent.invoke({"messages": [{"role": "user", "content": user_mg}]}
             ,{"configurable": {"thread_id": "1"}}, return_only_outputs=True)
    print(f"----: attached msg counts: {len(output['messages'])}")
    return output['messages'][-1].content

In [5]:
data = pd.read_json("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run /extracted_patterns/all_patterns.json")

In [6]:
data.head()

,Pattern Name,Problem,Context,Solution,Result,Related Patterns,Category,Uses,Thinking
0,External Knowledge Augmentation,Large Language Models (LLMs) are bounded by th...,LLMs relying on fixed and parametric knowledge...,Augment LLMs with the capability to access ext...,LLMs can surpass traditional knowledge limitat...,"Domain-Specific Tool Integration, Robust Tool-...","Knowledge & Reasoning, Tools Integration","Accessing contemporary information, retrieving...",This pattern directly addresses a core limitat...
1,Domain-Specific Tool Integration,"LLMs, trained on general knowledge, often exhi...",LLMs needing to perform tasks requiring deep e...,Employ specific external tools like online cal...,"Mitigates the expertise gap in LLMs, enhancing...","External Knowledge Augmentation, Task Automati...","Tools Integration, Knowledge & Reasoning","Performing complex calculations, solving equat...",This pattern focuses on overcoming the LLM's i...
2,Task Automation via Tools,LLMs are fundamentally language processors and...,Users requiring LLMs to perform real-world act...,Integrate LLMs with external task automation t...,LLMs can facilitate the execution of external ...,"Task Decomposition and Planning, Parameter Ext...","Agentic AI, Tools Integration","Scheduling appointments, setting reminders, fi...",This pattern enables LLMs to act as agents in ...
3,Multimodal Interaction Augmentation,LLMs often struggle to consistently understand...,User interactions involving varied input types...,Deploy specialized tools like speech recogniti...,Improved understanding and response to a broad...,Tool-Augmented Response Synthesis,"AI–Human Interaction, Tools Integration","Understanding speech inputs, analyzing images,...",This pattern addresses the limitation of LLMs ...
4,Transparent Tool-Use Reasoning,The opaque 'black-box' nature of current LLMs ...,"LLM applications where interpretability, accou...",Utilize tool learning to enable LLMs to exhibi...,"More transparent LLM operations, allowing user...","Iterative Task Solving (with Feedback), Tool-A...","AI–Human Interaction, Agentic AI, LLM-specific","Explaining complex problem-solving steps, debu...",This pattern directly addresses a critical eth...


In [7]:
data = data.sort_values(by="Pattern Name").reset_index(drop=True)

In [8]:
pattern_data = data.drop(columns=["Thinking","Uses","Category","Related Patterns"], errors="ignore")

In [ ]:
generate_description(pattern_data.iloc[1].to_dict())

In [ ]:
generate_description(pattern_data.iloc[1].to_dict())

In [9]:
pattern_descriptions = []
for _, row in pattern_data.iterrows():
    log_file = open("./logs/pattern_keywords_log.txt", "a")
    t = time()
    print(f"Generating keywords for pattern {_+1}/{len(pattern_data)}", end="  -  ")
    while True:
        out = generate_description(row.to_dict())
        parsed_output = parse_json_safe(out,delimiter="{}")
        description = parsed_output.get("keywords", "")
        thinking = parsed_output.get("thinking", "")
        if description:
            break
        print("Retrying...")
    pattern_descriptions.append(description)
    print(f"Generated in {time() - t:.2f} seconds")
    log_file.write(f"Pattern Name: {row['Pattern Name']}\n")
    log_file.write(f"{json.dumps(parsed_output, indent=2)}\n")
    log_file.write("-"*50 + "\n")
    log_file.close()

Generating keywords for pattern 1/349  -  ----: attached msg counts: 2
Generated in 8.74 seconds
Generating keywords for pattern 2/349  -  ----: attached msg counts: 4
Generated in 5.50 seconds
Generating keywords for pattern 3/349  -  ----: attached msg counts: 6
Generated in 3.37 seconds
Generating keywords for pattern 4/349  -  ----: attached msg counts: 8
Generated in 2.56 seconds
Generating keywords for pattern 5/349  -  ----: attached msg counts: 10
Generated in 3.17 seconds
Generating keywords for pattern 6/349  -  ----: attached msg counts: 12
Generated in 6.04 seconds
Generating keywords for pattern 7/349  -  ----: attached msg counts: 14
Generated in 4.71 seconds
Generating keywords for pattern 8/349  -  ----: attached msg counts: 16
Generated in 4.81 seconds
Generating keywords for pattern 9/349  -  ----: attached msg counts: 18
Generated in 4.71 seconds
Generating keywords for pattern 10/349  -  ----: attached msg counts: 20
Generated in 8.82 seconds
Generating keywords for

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count, limit: 1000000
Please retry in 57.29000961s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count"
  quota_id: "GenerateContentPaidTierInputTokensPerModelPerMinute"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 10000

----: attached msg counts: 280
Generated in 9.08 seconds
Generating keywords for pattern 141/349  -  ----: attached msg counts: 282
Generated in 3.47 seconds
Generating keywords for pattern 142/349  -  ----: attached msg counts: 284
Generated in 6.25 seconds
Generating keywords for pattern 143/349  -  ----: attached msg counts: 286
Generated in 5.43 seconds
Generating keywords for pattern 144/349  -  ----: attached msg counts: 288
Generated in 7.58 seconds
Generating keywords for pattern 145/349  -  ----: attached msg counts: 290
Generated in 4.30 seconds
Generating keywords for pattern 146/349  -  ----: attached msg counts: 292
Generated in 6.75 seconds
Generating keywords for pattern 147/349  -  ----: attached msg counts: 294
Generated in 4.92 seconds
Generating keywords for pattern 148/349  -  ----: attached msg counts: 296
Generated in 5.01 seconds
Generating keywords for pattern 149/349  -  ----: attached msg counts: 298
Generated in 6.86 seconds
Generating keywords for pattern 15

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count, limit: 1000000
Please retry in 57.468226461s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count"
  quota_id: "GenerateContentPaidTierInputTokensPerModelPerMinute"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1000

----: attached msg counts: 300
Generated in 9.34 seconds
Generating keywords for pattern 151/349  -  ----: attached msg counts: 302
Generated in 7.25 seconds
Generating keywords for pattern 152/349  -  ----: attached msg counts: 304
Generated in 6.04 seconds
Generating keywords for pattern 153/349  -  ----: attached msg counts: 306
Generated in 4.10 seconds
Generating keywords for pattern 154/349  -  ----: attached msg counts: 308
Generated in 5.56 seconds
Generating keywords for pattern 155/349  -  ----: attached msg counts: 310
Generated in 5.60 seconds
Generating keywords for pattern 156/349  -  ----: attached msg counts: 312
Generated in 5.02 seconds
Generating keywords for pattern 157/349  -  ----: attached msg counts: 314
Generated in 6.23 seconds
Generating keywords for pattern 158/349  -  ----: attached msg counts: 316
Generated in 4.63 seconds
Generating keywords for pattern 159/349  -  

...

----: attached msg counts: 682
Generated in 23.07 seconds
Generating keywords for pattern 342/349  -  ----: attached msg counts: 684
Generated in 8.26 seconds
Generating keywords for pattern 343/349  -  ----: attached msg counts: 686
Generated in 8.21 seconds
Generating keywords for pattern 344/349  -  

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count, limit: 1000000
Please retry in 27.844442544s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count"
  quota_id: "GenerateContentPaidTierInputTokensPerModelPerMinute"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1000

----: attached msg counts: 688
Generated in 51.28 seconds
Generating keywords for pattern 345/349  -  ----: attached msg counts: 690
Generated in 7.48 seconds
Generating keywords for pattern 346/349  -  ----: attached msg counts: 692
Generated in 7.14 seconds
Generating keywords for pattern 347/349  -  

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count, limit: 1000000
Please retry in 23.740770035s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_paid_tier_input_token_count"
  quota_id: "GenerateContentPaidTierInputTokensPerModelPerMinute"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1000

----: attached msg counts: 694
Generated in 48.89 seconds
Generating keywords for pattern 348/349  -  ----: attached msg counts: 696
Generated in 19.29 seconds
Generating keywords for pattern 349/349  -  ----: attached msg counts: 698
Generated in 8.93 seconds


In [ ]:
data['Description'] = pattern_descriptions
data.to_csv("descripted_patterns.csv", index=False)

### Using Vector DB

In [6]:
pattern_cols = ["Pattern Name", "Problem", "Solution", "Context", "Uses", "Result"]

In [68]:
embedding_data = pd.read_csv("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run /pattern_embeddings.csv")
embeddings = embedding_data.drop(columns=pattern_cols,errors="ignore")
embedding_data["Description"] = ["null"] * len(embedding_data)

In [122]:
def get_nearest_patterns(pattern_data: pd.DataFrame, top_k: int = 1) -> dict:
    distances = cosine_distances(pattern_data.drop(columns=pattern_cols, errors='ignore').values.reshape(1, -1), embeddings.values)
    distances[0][pattern_data.name] = np.inf
    closest_indices = np.argsort(distances)
    # discripted_closest_indices = [idx for idx in closest_indices[0] if embedding_data.iloc[idx]["Description"] != "null"][:top_k]
    closest_indices = closest_indices[0][:top_k]
    closest_patterns = []
    for i in closest_indices:
        closest_patterns.append({
            "Pattern Name": embedding_data.iloc[i]["Pattern Name"],
            "Solution": embedding_data.iloc[i]["Solution"],
            "Description": embedding_data.iloc[i]["Description"]
        })
        
    return closest_patterns
    

In [123]:
get_nearest_patterns(embeddings.iloc[1], top_k=3)

[{'Pattern Name': 'Tool Augmentation',
  'Solution': "Enhance LLMs' capabilities by integrating them with external specialized tools such as retrieval systems, math tools (e.g., WolframAlpha), code interpreters (e.g., Python, SQL), and structured databases.",
  'Description': 'null'},
 {'Pattern Name': 'Tool Use / Tool Augmentation',
  'Solution': "Equip the language agent with a 'Toolbox' of external tools (e.g., search engines, calculators, APIs like FlightSearch, CitySearch, RestaurantSearch, DistanceMatrix, AccommodationSearch, AttractionSearch) and the ability to select and use the appropriate tool based on the current task and context. The agent formulates tool calls and processes their observations.",
  'Description': 'null'},
 {'Pattern Name': 'External Knowledge Augmentation',
  'Solution': 'Augment LLMs with the capability to access external tools such as search engines, databases, and knowledge graphs to dynamically acquire and integrate external knowledge.',
  'Description'

In [75]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

In [117]:
prompt_template = """\
ROLE:
You are an expert AI design pattern analyst, specialized in identifying, classifying, and summarizing AI system design patterns such as RAG (Retrieval-Augmented Generation), Prompt Chaining, Vector Indexing, Fine-tuning, and RLHF (Reinforcement Learning from Human Feedback).

---

INPUT:
You will be given:
1. A current AI-related design pattern (e.g., “RAG”, “Prompt Chaining”, “Vector Indexing”, etc.).
2. A list of similar patterns, each containing:
   - Pattern name
   - Description
   - Keywords (optional)

---

TASK:
Generate a concise, factual, and contextually accurate representation of the current AI design pattern as comma-separated keywords that describe its purpose, function, architecture, workflow, and core elements.

Analyze the relationship between the current pattern and the given similar patterns, and decide whether to reuse, extend, or generate new keywords according to the rules below.

---

OUTPUT FORMAT (JSON):
  "keywords": "keyword1, keyword2, keyword3, keyword4, keyword5, keyword6, keyword7, keyword8, keyword9, keyword10, keyword11, keyword12, keyword13, keyword14, keyword15",
  "thinking": "Explain briefly how many similar patterns were valid and how you decided to reuse or generate keywords."

---

RULES & LOGIC:

GENERAL RULES:
1. Always output exactly 15 keywords.
2. Keywords must be factual, relevant, and grounded in known AI terminology.
3. Do not invent or speculate new terms.
4. Avoid duplicate, vague, or overly generic keywords.
5. Maintain consistent terminology across all patterns.
6. Ignore patterns with empty or null keyword fields.

---

REUSE & EXTENSION LOGIC:

Case 1: Identical or Previously Grouped Pattern
→ If the current pattern is identical to or already grouped with a previous one:
✅ Reuse all 15 keywords exactly as-is (no changes, no additions).

Case 2: Strong Similarity
→ If the current pattern is very similar to an existing one:
✅ Reuse 12 existing keywords from the most relevant similar pattern.
➕ Add 3 new keywords that capture unique features of the current pattern.

Case 3: Moderate Similarity
→ If the pattern shares some overlap but not fully identical:
✅ Reuse 13 existing keywords.
➕ Add 2 new keywords to emphasize its distinct characteristics.

Case 4: Weak or No Similarity
→ If none of the given patterns are truly related:
✅ Ignore them all and generate 15 new, meaningful, and relevant keywords based solely on the current pattern.

---

VALIDATION GUIDELINES:
Before reusing any keywords:
1. Confirm similarity logically based on purpose, structure, and domain.
2. Ignore any pattern whose description or keywords are conceptually irrelevant.
3. If the pattern can be grouped with another, reuse the exact previous keywords — no alterations allowed.

---

FINAL OUTPUT REQUIREMENTS:
Include both fields:
1. "keywords" → The final 15 selected keywords
2. "thinking" → A very short reasoning explaining how many similar patterns were validated and how the keywords were reused or newly generated. (Max 20 words)

---

FINAL INSTRUCTION:
Now analyze the following input and generate your output accordingly.

Current Pattern:
{pattern}

Similar Patterns:
Solutions | Keywords
{similar_patterns}
"""

In [118]:
def write_log(pattern_name: str, output: str):
    log_file = open("./logs/pattern_keywords_log_verctor_version_v2.txt", "a")
    log_file.write(f"Pattern Name: {pattern_name}")
    log_file.write(f"{json.dumps(output, indent=2)}\n")
    log_file.write("-"*50 + "\n")
    log_file.close()

In [120]:
def generate_description_with_context(pattern: dict, similar_patterns: list[dict])->str:
    user_mg = prompt_template.format(
        pattern=json.dumps(pattern, indent=2),
        similar_patterns="\n".join([f"{p['Solution']} -> {p['Description']}" for p in similar_patterns])
    )
    output = llm.invoke(user_mg)
    parsed_output = parse_json_safe(output.content, delimiter="{}")
    if parsed_output.get("keywords", "") == "":
        return generate_description_with_context(pattern, similar_patterns)
    
    write_log(pattern['Pattern Name'], parsed_output)
    return parsed_output

In [124]:
generate_description_with_context(embedding_data[pattern_cols].iloc[1].to_dict(), get_nearest_patterns(embeddings.iloc[1], top_k=3))

{'keywords': 'LLM augmentation, tool integration, API interaction, domain-specific expertise, computational capabilities, problem-solving augmentation, external knowledge, dynamic retrieval, database querying, factual accuracy, hallucination mitigation, data grounding, pretraining boundary extension, information synthesis, contextual relevance',
 'thinking': 'One similar pattern was valid. Reused 12 keywords and generated 3 new ones due to strong similarity.'}

In [125]:
embedding_data["Description"] = ["null"] * len(embedding_data)
for i in range(len(embedding_data)):
    print(f"Generating keywords for pattern {i+1}/{len(embedding_data)}", end="  -  ")
    t = time()
    similar_patterns = get_nearest_patterns(embeddings.iloc[i], top_k=25)
    out = generate_description_with_context(embedding_data[pattern_cols].iloc[i].to_dict(), similar_patterns)
    embedding_data.at[i, "Description"] = out.get("keywords", "")
    print(f"Generated in {time() - t:.2f} seconds")

Generating keywords for pattern 1/349  -  Generated in 11.47 seconds
Generating keywords for pattern 2/349  -  Generated in 22.53 seconds
Generating keywords for pattern 3/349  -  

KeyboardInterrupt: 

In [98]:
embedding_data.to_csv("descripted_patterns_vector_version.csv", index=False)

### Using Paramter scoring

In [129]:
patterns = pd.read_json("/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/outputs/prompt & rag/20251028_085918 - Run /extracted_patterns/all_patterns.json")

In [148]:
evaluation_prompt = """\
Your ROLE:
You are an expert AI design pattern analyst, specialized in identifying, classifying, and scoring AI system design patterns such as RAG (Retrieval-Augmented Generation), Prompt Chaining, Vector Indexing, Fine-tuning, and RLHF (Reinforcement Learning from Human Feedback).

TASK:
Given a current AI design pattern, you need to evaluate them by given criteria.

Evaluation Criteria:
| No | Parameter                       | Type                                                                                                                           | Description                                                        |
| -- | ------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ | ------------------------------------------------------------------ |
| 1  | `llm_role`                      | categorical (`planner`, `reasoner`, `executor`, `generator`, `analyzer`, `none`)                                               | Defines the main role of the LLM in the pattern.                   |
| 2  | `involves_stage`                | categorical (`training`, `inference`, `deployment`, `entire_pipeline`)                                                         | Where the pattern applies in the ML lifecycle.                     |
| 3  | `category_alignment`            | categorical (`agentic_ai`, `reasoning`, `knowledge`, `planning`, `interaction`, `mlops`, `generative_ai`, `tools_integration`) | Broad category label alignment.                                    |
| 4  | `is_reasoning_focused`          | numeric (0–10)                                                                                                                 | Extent to which the pattern enhances or depends on reasoning.      |
| 5  | `solves_hallucination`          | numeric (0–10)                                                                                                                 | Effectiveness at reducing hallucination.                           |
| 6  | `enhances_factuality`           | numeric (0–10)                                                                                                                 | Improves factual accuracy.                                         |
| 7  | `improves_trust`                | numeric (0–10)                                                                                                                 | Improves interpretability, transparency, or user trust.            |
| 8  | `is_tool_based`                 | boolean                                                                                                                        | Whether the pattern explicitly integrates external tools/APIs.     |
| 9  | `is_knowledge_driven`           | boolean                                                                                                                        | Whether it involves external knowledge (e.g., KG, RAG).            |
| 10 | `is_self_reflective`            | boolean                                                                                                                        | Whether it involves self-evaluation or feedback (e.g., autorater). |
| 11 | `is_agentic`                    | boolean                                                                                                                        | If it involves autonomous reasoning, planning, or acting.          |
| 12 | `involves_retrieval`            | boolean                                                                                                                        | Uses retrieval mechanisms.                                         |
| 13 | `involves_generation`           | boolean                                                                                                                        | Produces new textual or code outputs.                              |
| 14 | `uses_feedback_loop`            | boolean                                                                                                                        | Employs feedback from previous steps or user/tool output.          |
| 15 | `has_planning_phase`            | boolean                                                                                                                        | Includes explicit task decomposition or planning.                  |
| 16 | `involves_multi_step_reasoning` | boolean                                                                                                                        | Performs multi-hop or multi-phase reasoning.                       |
| 17 | `improves_efficiency`           | numeric (0–10)                                                                                                                 | Reduces computational or inference cost.                           |
| 18 | `improves_accuracy`             | numeric (0–10)                                                                                                                 | Improves final model correctness.                                  |
| 19 | `reduces_latency`               | numeric (0–10)                                                                                                                 | Makes processing faster.                                           |
| 20 | `supports_modularity`           | boolean                                                                                                                        | Can integrate as a plug-and-play component.                        |
| 21 | `data_dependency_level`         | numeric (0–10)                                                                                                                 | Reliance on external or preprocessed data.                         |
| 22 | `context_awareness`             | numeric (0–10)                                                                                                                 | Ability to use contextual information effectively.                 |
| 23 | `is_interpretable`              | boolean                                                                                                                        | Provides explanations or interpretable outputs.                    |
| 24 | `enhances_user_interaction`     | numeric (0–10)                                                                                                                 | Improves dialogue or user control.                                 |
| 25 | `enables_abstention`            | boolean                                                                                                                        | Can abstain or refuse uncertain outputs.                           |
| 26 | `supports_autonomy`             | boolean                                                                                                                        | Acts independently without user supervision.                       |
| 27 | `robustness_level`              | numeric (0–10)                                                                                                                 | Stability against noisy or adversarial inputs.                     |
| 28 | `adaptability_level`            | numeric (0–10)                                                                                                                 | Ability to adapt dynamically to new contexts.                      |
| 29 | `uses_external_api`             | boolean                                                                                                                        | Calls or integrates with APIs/databases.                           |
| 30 | `has_validation_mechanism`      | boolean                                                                                                                        | Includes internal or external validation (e.g., verifier).         |
| 31 | `supports_multi_modal`          | boolean                                                                                                                        | Processes non-text data (image, speech, etc.).                     |
| 32 | `is_llm_collaborative`          | boolean                                                                                                                        | Involves multiple LLMs working together.                           |
| 33 | `is_llm_created_tool`           | boolean                                                                                                                        | LLM generates tools or models automatically.                       |
| 34 | `improves_data_quality`         | numeric (0–10)                                                                                                                 | Enhances or filters input data or context.                         |
| 35 | `reduces_bias`                  | numeric (0–10)                                                                                                                 | Reduces social, factual, or algorithmic bias.                      |
| 36 | `safety_focus`                  | numeric (0–10)                                                                                                                 | Addresses security or ethical concerns.                            |
| 37 | `training_needed`               | boolean                                                                                                                        | Requires additional training or fine-tuning.                       |
| 38 | `tuning_type`                   | categorical (`none`, `lora`, `instruction_tuning`, `finetune_full`, `prompt_based`)                                            | What kind of tuning is used.                                       |
| 39 | `evaluation_capability`         | boolean                                                                                                                        | Evaluates model outputs or contexts (e.g., autorater).             |
| 40 | `human_in_loop`                 | boolean                                                                                                                        | Involves human supervision, feedback, or UI interaction.           |
| 41 | `automation_level`              | numeric (0–10)                                                                                                                 | Degree of automatic operation or decision-making.                  |
| 42 | `generalization_score`          | numeric (0–10)                                                                                                                 | How broadly applicable across tasks/domains.                       |
| 43 | `context_sufficiency_awareness` | boolean                                                                                                                        | Detects whether provided context is sufficient (e.g., Autorater).  |
| 44 | `uncertainty_management`        | boolean                                                                                                                        | Manages uncertainty/confidence estimation.                         |
| 45 | `faithfulness_score`            | numeric (0–10)                                                                                                                 | Degree of alignment between reasoning steps and truth.             |
| 46 | `reasoning_structure`           | categorical (`chain`, `tree`, `graph`, `none`)                                                                                 | Reasoning architecture type.                                       |
| 47 | `retrieval_type`                | categorical (`single_step`, `multi_step`, `adaptive`, `none`)                                                                  | Retrieval method style.                                            |
| 48 | `output_type`                   | categorical (`text`, `plan`, `code`, `graph`, `decision`, `explanation`)                                                       | Nature of final output.                                            |
| 49 | `pattern_complexity`            | numeric (0–10)                                                                                                                 | Conceptual and implementation complexity.                          |
| 50 | `reusability_score`             | numeric (0–10)                                                                                                                 | Ease of reusing or combining with other patterns.                  |

* Do not give any parameter as a List type values as a LLM output. Only give single categorical value.

EVALUATION OUTPUT:
Provide a JSON object with the parameter names as keys and their evaluated values according to the criteria above

-----

Pattern:
{pattern}
"""

In [134]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

In [135]:
eval_parameters = [ "llm_role", "involves_stage", "category_alignment", "is_reasoning_focused", "solves_hallucination", "enhances_factuality", "improves_trust", "is_tool_based", "is_knowledge_driven", "is_self_reflective", "is_agentic", "involves_retrieval", "involves_generation", "uses_feedback_loop", "has_planning_phase", "involves_multi_step_reasoning", "improves_efficiency", "improves_accuracy", "reduces_latency", "supports_modularity", "data_dependency_level", "context_awareness", "is_interpretable", "enhances_user_interaction", "enables_abstention", "supports_autonomy", "robustness_level", "adaptability_level", "uses_external_api", "has_validation_mechanism", "supports_multi_modal", "is_llm_collaborative", "is_llm_created_tool", "improves_data_quality", "reduces_bias", "safety_focus", "training_needed", "tuning_type", "evaluation_capability", "human_in_loop", "automation_level", "generalization_score", "context_sufficiency_awareness", "uncertainty_management", "faithfulness_score", "reasoning_structure", "retrieval_type", "output_type", "pattern_complexity", "reusability_score" ]

In [149]:
def generate_evaluation(pattern: dict)->str:
    user_mg = evaluation_prompt.format(
        pattern=json.dumps(pattern, indent=2),
    )
    output = llm.invoke(user_mg)
    parsed_output = parse_json_safe(output.content, delimiter="{}")

    key_validation = True

    for key in eval_parameters:
        if key not in parsed_output:
            key_validation = False
            break
    
    if key_validation == False:
        print(f"Validation failed for pattern: {pattern['Pattern Name']}. Retrying evaluation.")
        return generate_evaluation(pattern)
    
    write_log(pattern['Pattern Name'], parsed_output)
    return parsed_output

In [150]:
evaluated_patterns = []
for pattern in patterns.to_dict(orient="records"):
    t = time()
    print(f"Evaluating pattern({len(evaluated_patterns)+1}/{len(patterns)}): {pattern['Pattern Name']} - ",end="")
    evaluation = generate_evaluation(pattern)
    pattern.update(evaluation)
    evaluated_patterns.append(pattern)
    print(f"Time taken: {round(time() - t)} seconds - Estimated total time: {round((time() - t) * (len(patterns) - len(evaluated_patterns)) / 60)} minutes")

Evaluating pattern(1/349): External Knowledge Augmentation - Time taken: 29 seconds - Estimated total time: 166 minutes
Evaluating pattern(2/349): Domain-Specific Tool Integration - Time taken: 20 seconds - Estimated total time: 114 minutes
Evaluating pattern(3/349): Task Automation via Tools - Time taken: 21 seconds - Estimated total time: 119 minutes
Evaluating pattern(4/349): Multimodal Interaction Augmentation - Time taken: 20 seconds - Estimated total time: 117 minutes
Evaluating pattern(5/349): Transparent Tool-Use Reasoning - Time taken: 18 seconds - Estimated total time: 104 minutes
Evaluating pattern(6/349): Robust Tool-Augmented Processing - Time taken: 20 seconds - Estimated total time: 116 minutes
Evaluating pattern(7/349): Iterative Task Solving (with Feedback) - Time taken: 20 seconds - Estimated total time: 115 minutes
Evaluating pattern(8/349): Task Decomposition and Planning - Time taken: 22 seconds - Estimated total time: 127 minutes
Evaluating pattern(9/349): Tool Re

In [151]:
len(evaluated_patterns)


349

In [152]:
pd.DataFrame(evaluated_patterns).to_csv("./datasets/evaluated_patterns_v1.csv", index=False)